In [66]:
%reload_ext autoreload
%autoreload 2

from datetime import datetime, timezone
import pytz
from time import sleep
import numpy as np

from compute_ror_prob import compute_floor_price
from functions import calculate_l_v
from trading_api_functions import get_recent_btc_data, get_kalshi_btc_spreads

In [ ]:
bankroll = 10
risk_threshold = 0.005
W_star = np.array([[9.986698006428435, 9.263552603133919], [3.4524303392799243, 3.194742074941929], [4.329087071447457, 6.26646244471938], [7.346263185019447, 5.126563640350223], [5.236380091612666, 0.3741145148544866], [8.787414902561695, 2.4931340280198295], [4.5660909920929535, 8.462778051128941], [6.958557373886825, 5.910271393802765], [6.189210279401085, 2.596576364569244], [7.426106870329473, 7.092833445853189]])
btc_df = get_recent_btc_data()
print(btc_df)
lookahead_time = 2 * 3600 # Keep to 2 hours because prediction falls off after that

      Timestamp                Date        Low       High       Open  \
349  1749591180 2025-06-10 16:33:00  109828.62  109857.68  109849.20   
348  1749591240 2025-06-10 16:34:00  109812.62  109832.61  109832.61   
347  1749591300 2025-06-10 16:35:00  109819.32  109913.36  109819.33   
346  1749591360 2025-06-10 16:36:00  109911.71  109969.63  109911.72   
345  1749591420 2025-06-10 16:37:00  109774.41  109954.36  109941.06   
..          ...                 ...        ...        ...        ...   
4    1749611880 2025-06-10 22:18:00  109745.00  109786.87  109745.05   
3    1749611940 2025-06-10 22:19:00  109750.50  109796.15  109755.02   
2    1749612000 2025-06-10 22:20:00  109779.25  109798.58  109794.23   
1    1749612060 2025-06-10 22:21:00  109780.69  109806.76  109783.19   
0    1749612120 2025-06-10 22:22:00  109793.53  109813.19  109793.55   

         Close    Volume       Price  
349  109832.59  1.329406  109843.150  
348  109819.33  2.033104  109822.615  
347  109911.72  5.

In [104]:
deltas = [5, 25, 50, 100, 120] # in minutes
ps = [-0.005, -0.01, -0.015, -0.02, -0.025]
for d in deltas:
    for p in ps:
        l, v = calculate_l_v(W_star, 3, d, p)
        print(f"Delta: {d}, P: {p}, Window Size: {l}, Nu: {v}")


Delta: 5, P: -0.005, Window Size: 10, Nu: 9.437706797388191
Delta: 5, P: -0.01, Window Size: 10, Nu: 9.469369660157405
Delta: 5, P: -0.015, Window Size: 10, Nu: 9.50117322841228
Delta: 5, P: -0.02, Window Size: 10, Nu: 9.533122821777903
Delta: 5, P: -0.025, Window Size: 10, Nu: 9.565223759879355
Delta: 25, P: -0.005, Window Size: 11, Nu: 10.261234851458777
Delta: 25, P: -0.01, Window Size: 11, Nu: 10.294473240068635
Delta: 25, P: -0.015, Window Size: 11, Nu: 10.327873972300528
Delta: 25, P: -0.02, Window Size: 11, Nu: 10.361442367779539
Delta: 25, P: -0.025, Window Size: 11, Nu: 10.395183746130753
Delta: 50, P: -0.005, Window Size: 13, Nu: 12.134236093375543
Delta: 50, P: -0.01, Window Size: 13, Nu: 12.171752589049412
Delta: 50, P: -0.015, Window Size: 13, Nu: 12.209458476015778
Delta: 50, P: -0.02, Window Size: 13, Nu: 12.247359073899727
Delta: 50, P: -0.025, Window Size: 13, Nu: 12.285459702326342
Delta: 100, P: -0.005, Window Size: 20, Nu: 20.43692251519925
Delta: 100, P: -0.01, Win

In [108]:
window_size = None
nu = None
ror_price = None
while True:
    kalshi_df = get_kalshi_btc_spreads(lookahead_time)
    close_time = kalshi_df.loc[0]['close_time']
    kalshi_df = kalshi_df[kalshi_df['close_time'] == close_time]
    print(kalshi_df.head())

    print(btc_df.tail())

    dt_now = datetime.now(pytz.timezone("America/Chicago"))
    delta_min = (close_time - dt_now).total_seconds() / 60.0
    print(delta_min)

    p_0 = btc_df.loc[len(btc_df) - 1]['Price']
    ror_price = p_0

    # Simple loop to determine gbm params and ror_price
    while True:
        new_window_size, new_nu = calculate_l_v(W_star, 3, delta_min, (ror_price - p_0) / p_0)

        btc_win = btc_df.tail(new_window_size)
        new_ror_price = compute_floor_price(delta_min, btc_win, p_0, new_nu, risk_threshold)
        print(f"Window Size- Old: {window_size}, New: {new_window_size}")
        print(f"Nu- Old: {nu}, New: {round(new_nu, 3)}")
        print(f"ROR Price- Old: {ror_price}, New: {new_ror_price}")
        if window_size == new_window_size and round(nu, 3) == round(new_nu, 3) and round(ror_price, 3) == round(new_ror_price, 3):
            window_size = new_window_size
            nu = new_nu
            ror_price = new_ror_price
            break
        else:
            window_size = new_window_size
            nu = new_nu
            ror_price = new_ror_price

    # sleep(1)
    break

# ror_price = compute_floor_price(delta, data)
# window_size, nu = calculate_l_v(W, 3, delta, )

            subtitle  no_ask                close_time  ceil_price
0  $115,000 or above     100 2025-06-10 23:00:00-05:00      115000
1  $114,750 or above     100 2025-06-10 23:00:00-05:00      114750
2  $114,500 or above     100 2025-06-10 23:00:00-05:00      114500
3  $114,250 or above     100 2025-06-10 23:00:00-05:00      114250
4  $114,000 or above     100 2025-06-10 23:00:00-05:00      114000
    Timestamp                Date        Low       High       Open      Close  \
4  1749611880 2025-06-10 22:18:00  109745.00  109786.87  109745.05  109755.41   
3  1749611940 2025-06-10 22:19:00  109750.50  109796.15  109755.02  109794.23   
2  1749612000 2025-06-10 22:20:00  109779.25  109798.58  109794.23  109783.19   
1  1749612060 2025-06-10 22:21:00  109780.69  109806.76  109783.19  109791.91   
0  1749612120 2025-06-10 22:22:00  109793.53  109813.19  109793.55  109813.19   

     Volume       Price  
4  1.459721  109765.935  
3  0.638110  109773.325  
2  1.978052  109788.915  
1  2.37

<string>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<string>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<string>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
